📊 Modern Data Pipeline for Retail Analytics

This project simulates a modern data pipeline using a Medallion Architecture approach:

Bronze Layer → Raw data ingestion
Silver Layer → Data cleaning & validation
Gold Layer → Business-level transformations
🔧 Key Features:
End-to-end data pipeline using Python
Data quality validation checks
Modular transformation logic (dbt-inspired)
Business KPI generation (revenue, orders)


In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

data = {
    "order_id": range(1, 1001),
    "product": np.random.choice(["Toy A", "Toy B", "Toy C"], 1000),
    "region": np.random.choice(["US", "EU", "APAC"], 1000),
    "sales": np.random.randint(10, 500, 1000),
    "quantity": np.random.randint(1, 10, 1000),
    "order_date": pd.date_range(start="2024-01-01", periods=1000)
}

df_raw = pd.DataFrame(data)

df_raw.head()

,order_id,product,region,sales,quantity,order_date
0,1,Toy C,APAC,247,3,2024-01-01
1,2,Toy A,APAC,365,1,2024-01-02
2,3,Toy C,APAC,182,8,2024-01-03
3,4,Toy C,APAC,401,5,2024-01-04
4,5,Toy A,US,230,3,2024-01-05


In [3]:
df_staging = df_raw.copy()

# Handle missing values
df_staging["sales"] = df_staging["sales"].fillna(0)
df_staging["quantity"] = df_staging["quantity"].fillna(1)

df_staging.head()

,order_id,product,region,sales,quantity,order_date
0,1,Toy C,APAC,247,3,2024-01-01
1,2,Toy A,APAC,365,1,2024-01-02
2,3,Toy C,APAC,182,8,2024-01-03
3,4,Toy C,APAC,401,5,2024-01-04
4,5,Toy A,US,230,3,2024-01-05


In [4]:
df_staging["revenue"] = df_staging["sales"] * df_staging["quantity"]

df_gold = df_staging.groupby("region").agg({
    "revenue": "sum",
    "order_id": "count"
}).reset_index()

df_gold.rename(columns={"order_id": "total_orders"}, inplace=True)

df_gold

,region,revenue,total_orders
0,APAC,437315,333
1,EU,430428,333
2,US,433720,334


In [5]:
# Null check
assert df_staging.isnull().sum().sum() == 0, "❌ Null values found!"

# Business rule checks
assert (df_staging["sales"] >= 0).all(), "❌ Negative sales found!"

print("✅ Data quality checks passed!")

✅ Data quality checks passed!


In [6]:
print("Total records:", len(df_staging))
print("Total revenue:", df_staging["revenue"].sum())
print("Regions:", df_staging["region"].unique())

Total records: 1000
Total revenue: 1301463
Regions: ['APAC' 'US' 'EU']


In [7]:
df_raw.to_csv("raw_sales.csv", index=False)
df_staging.to_csv("staging_sales.csv", index=False)
df_gold.to_csv("gold_sales_summary.csv", index=False)

In [8]:
!pip install pyspark

In [9]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Retail Data Pipeline") \
    .getOrCreate()

print("✅ Spark session started")

✅ Spark session started


In [10]:
import pandas as pd
import numpy as np

np.random.seed(42)

data = {
    "order_id": range(1, 1001),
    "product": np.random.choice(["Toy A", "Toy B", "Toy C"], 1000),
    "region": np.random.choice(["US", "EU", "APAC"], 1000),
    "sales": np.random.randint(10, 500, 1000),
    "quantity": np.random.randint(1, 10, 1000),
    "order_date": pd.date_range(start="2024-01-01", periods=1000)
}

df_pandas = pd.DataFrame(data)

# Convert to Spark DataFrame
df_bronze = spark.createDataFrame(df_pandas)

df_bronze.show(5)

+--------+-------+------+-----+--------+-------------------+
|order_id|product|region|sales|quantity|         order_date|
+--------+-------+------+-----+--------+-------------------+
|       1|  Toy C|  APAC|  247|       3|2024-01-01 00:00:00|
|       2|  Toy A|  APAC|  365|       1|2024-01-02 00:00:00|
|       3|  Toy C|  APAC|  182|       8|2024-01-03 00:00:00|
|       4|  Toy C|  APAC|  401|       5|2024-01-04 00:00:00|
|       5|  Toy A|    US|  230|       3|2024-01-05 00:00:00|
+--------+-------+------+-----+--------+-------------------+
only showing top 5 rows


In [11]:
from pyspark.sql.functions import col

df_silver = df_bronze.fillna({
    "sales": 0,
    "quantity": 1
})

df_silver.show(5)

+--------+-------+------+-----+--------+-------------------+
|order_id|product|region|sales|quantity|         order_date|
+--------+-------+------+-----+--------+-------------------+
|       1|  Toy C|  APAC|  247|       3|2024-01-01 00:00:00|
|       2|  Toy A|  APAC|  365|       1|2024-01-02 00:00:00|
|       3|  Toy C|  APAC|  182|       8|2024-01-03 00:00:00|
|       4|  Toy C|  APAC|  401|       5|2024-01-04 00:00:00|
|       5|  Toy A|    US|  230|       3|2024-01-05 00:00:00|
+--------+-------+------+-----+--------+-------------------+
only showing top 5 rows


In [12]:
from pyspark.sql.functions import sum as _sum, count

# Create revenue column
df_silver = df_silver.withColumn(
    "revenue", col("sales") * col("quantity")
)

# Aggregate
df_gold = df_silver.groupBy("region").agg(
    _sum("revenue").alias("total_revenue"),
    count("order_id").alias("total_orders")
)

df_gold.show()

+------+-------------+------------+
|region|total_revenue|total_orders|
+------+-------------+------------+
|  APAC|       437315|         333|
|    US|       433720|         334|
|    EU|       430428|         333|
+------+-------------+------------+



In [13]:
# Check nulls
null_count = df_silver.filter(
    col("sales").isNull() | col("quantity").isNull()
).count()

assert null_count == 0, "❌ Null values found!"

# Check negative values
negative_count = df_silver.filter(col("sales") < 0).count()

assert negative_count == 0, "❌ Negative sales found!"

print("✅ Data quality checks passed!")


✅ Data quality checks passed!


In [14]:
print("Total records:", df_silver.count())
print("Total revenue:", df_silver.selectExpr("sum(revenue)").collect()[0][0])

Total records: 1000
Total revenue: 1301463
